# FHI-aims Trajectory and Force Merger for OVITO

This script extracts atomic forces from an FHI-aims output file and combines them with the corresponding atomic coordinates stored in an XYZ trajectory. For each Molecular Dynamics step, the atomic positions and forces are merged into an extended XYZ file containing the following properties:
- Atomic species
- Cartesian coordinates (x, y, z)
- Force components (Fx, Fy, Fz)
The resulting trajectory can be directly loaded into OVITO for visualization and analysis of atomic structures, force distributions, and dynamical processes throughout the simulation.
Input files:
- trajectory.xyz : Atomic coordinates for all MD steps.
- salida.out     : FHI-aims output containing atomic forces.
Output file:
- trajectory_forces.xyz : Extended XYZ trajectory with force information.


In [1]:
#!/usr/bin/env python3
import re

xyz_file = "trajectory.xyz"
aims_out = "salida.out"
output_xyz = "trajectory_forces.xyz"

# --------------------------------------------------
# 1. Leer fuerzas de aims.out
# --------------------------------------------------
forces_steps = []
current_forces = []
reading = False

with open(aims_out, "r") as f:
    for line in f:
        if "Total atomic forces (unitary forces cleaned)" in line:
            current_forces = []
            reading = True
            continue

        if reading:
            if line.strip().startswith("|"):
                parts = line.split()
                fx = float(parts[2])
                fy = float(parts[3])
                fz = float(parts[4])
                current_forces.append((fx, fy, fz))
            else:
                if current_forces:
                    forces_steps.append(current_forces)
                reading = False

# --------------------------------------------------
# 2. Leer trayectoria XYZ
# --------------------------------------------------
with open(xyz_file, "r") as f:
    xyz_lines = f.readlines()

out = open(output_xyz, "w")

i = 0
step = 0

while i < len(xyz_lines):
    natoms = int(xyz_lines[i].strip())
    atoms = xyz_lines[i+2:i+2+natoms]

    forces = forces_steps[step]

    out.write(f"{natoms}\n")
    out.write(
        "Properties=species:S:1:pos:R:3:force:R:3 "
        f"Step={step}\n"
    )

    for j, atom in enumerate(atoms):
        s, x, y, z = atom.split()[:4]
        fx, fy, fz = forces[j]
        out.write(
            f"{s} {x} {y} {z} "
            f"{fx:.8e} {fy:.8e} {fz:.8e}\n"
        )

    i += natoms + 2
    step += 1

out.close()

print("✔ Archivo OVITO generado:", output_xyz)

✔ Archivo OVITO generado: trajectory_forces.xyz
